In [1]:
# libriaries used are loaded

from google.colab import drive
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from pprint import pprint
from __future__ import absolute_import, division, print_function, unicode_literals
import collections
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.compat.v1.keras.layers import CuDNNGRU
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, Embedding, GRU, LSTM, RNN, SpatialDropout1D, Bidirectional
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
print(stopwords.words('english'))
from nltk.tokenize import word_tokenize
from tensorflow.keras.callbacks import EarlyStopping
import pickle

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'ea

In [2]:
df_test=pd.read_csv("https://raw.githubusercontent.com/avalmahsa/Fake-news-detection/main/x_test.csv")

In [3]:
## Import dataset
df1 = pd.read_csv('https://raw.githubusercontent.com/laxmimerit/fake-real-news-dataset/main/data/True.csv')
df1['label']=int(1)
df2= pd.read_csv('https://raw.githubusercontent.com/laxmimerit/fake-real-news-dataset/main/data/Fake.csv')
df2['label']=int(0)
df3=pd.read_csv("https://raw.githubusercontent.com/avalmahsa/Fake-news-detection/main/xy_train.csv")
df1=df1.drop(axis=1, index=None, columns="subject")
df2=df2.drop(axis=1, index=None, columns="subject")
df1=df1.drop(axis=1, index=None, columns="date")
df2=df2.drop(axis=1, index=None, columns="date")
df1["text"]=df1["title"]+" "+df1["text"]
df2["text"]=df2["title"]+" "+df2["text"]
df1=df1.drop(axis=1, index=None, columns="title")
df2=df2.drop(axis=1, index=None, columns="title")
df_train=pd.concat([df1,df2,df3]).reset_index(drop=True)


In [4]:
df_train=df_train.drop(axis=1, index=None, columns="id")

In [5]:
#df_test

In [6]:
x =df_train.text
y =df_train.label
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.2)

In [7]:
# maximum number of words from the resulting tokenized data which are to be used
vocab_size = 40000
max_len = 40
# build vocabulary from training set
tokenizer = Tokenizer(num_words=vocab_size)
# upadting the internal vocalulary based on the list of text, so it creates the vocabulary
# index based on word frequnecy so every word gets a unique interger value so lower integers 
# mean more frequent word.
tokenizer.fit_on_texts(x_train)
def _preprocess(list_of_text):
    # pads sequence to the same length (all sequences in a list to have the same length), it
    # does so by padding 0 in the beggining of each sequence until they have the same length as
    # the longest sequence. 
    return pad_sequences(
        # transforms each text in texts to a sequence of integers. It takes each word
        # in the text and replaces it with its corresponding integer value from the 
        # dictionary.
        tokenizer.texts_to_sequences(list_of_text),
        # takes in the pre-defined input (40) as maximum length of all sequences.
        maxlen=max_len,
        # does padding after each sequence
        padding='post',
    )
# padding is done inside: 
x_train = _preprocess(x_train)
x_valid = _preprocess(x_valid)
print(x_train.shape, y_train.shape)
print(x_valid.shape, y_valid.shape)

(83918, 40) (83918,)
(20980, 40) (20980,)


In [8]:
# Stop word removal before passing it to preprocessing
x =df_train.text
def filter_stop_words(x, stop_words):
    for i, sentence in enumerate(x):
        new_sent = [word for word in sentence.split() if word not in stop_words]
        x[i] = ' '.join(new_sent)
    return x
stop_words = set(stopwords.words("english"))
train_sentences = filter_stop_words(x, stop_words)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [9]:
# these match as we can see. It has 23 words and the rest is padded to match 40 which was a 
# predefined argument. 
print(x_train[:4])
pprint(tokenizer.sequences_to_texts(x_train[:4]))

[[  537   370     7    39   892  9314  4714 19864     1 23650   823 18278
      1 22486 24324   186  3039  1894  3557  2706     1 27378   186  2767
  16462    61  1122  3965     3   658     6   540     0     0     0     0
      0     0     0     0]
 [    7  2455    99  5080  5328     1   530   643    36    39    24 14605
    206     2   281     4   697     9     7    10 18875   409    35    93
      4  6489   939     5   135   574     7  1220     4  1396     5  5128
    530    52    15  2455]
 [ 2835 30747  3480  5573  5574  9076    16    19  7956   989    33    26
    113     1  9076    23  1745  7050     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0]
 [35635 21531 21532    51   160   207 18876  5402     5     4   725 22487
      2   393     1  2773  7348     1   102   891    10    30   122   356
    140  6845     0     0     0     0     0     0     0     0     0     0
      0     0     0     0]]
['7

In [10]:
# Bi_directional Recurrent LSTM
seq_in = keras.Input(batch_shape=(None, max_len))
embedded = keras.layers.Embedding(tokenizer.num_words, 100)(seq_in)
#averaged = tf.reduce_mean(embedded, axis=1)

bgru1 = Bidirectional(LSTM(units = 100, dropout = 0.2, recurrent_dropout = 0.2))(embedded)
do1 = Dropout(rate = 0.2)(bgru1)
d1 = keras.layers.Dense(100)(do1)
pred = keras.layers.Dense(1, activation='sigmoid')(d1)

model = keras.Model(
    inputs=seq_in,
    outputs=pred,
)

model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC']
)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(x_train,
                    y_train,
                    epochs=5,
                    batch_size=150,
                    validation_data=(x_valid, y_valid),
                    verbose=1)

print(x_train.shape, y_train.shape)
print(x_valid.shape, y_valid.shape)

Epoch 1/5
560/560 [==============================] - 240s 419ms/step - loss: 0.3225 - accuracy: 0.8471 - auc: 0.9344 - val_loss: 0.2657 - val_accuracy: 0.8843 - val_auc: 0.9576
Epoch 2/5
560/560 [==============================] - 236s 422ms/step - loss: 0.1872 - accuracy: 0.9212 - auc: 0.9779 - val_loss: 0.2805 - val_accuracy: 0.8790 - val_auc: 0.9572
Epoch 3/5
560/560 [==============================] - 234s 418ms/step - loss: 0.1200 - accuracy: 0.9467 - auc: 0.9892 - val_loss: 0.3419 - val_accuracy: 0.8774 - val_auc: 0.9504
Epoch 4/5
560/560 [==============================] - 234s 419ms/step - loss: 0.1168 - accuracy: 0.9453 - auc: 0.9870 - val_loss: 0.3999 - val_accuracy: 0.8655 - val_auc: 0.9438
Epoch 5/5
560/560 [==============================] - 234s 418ms/step - loss: -0.0127 - accuracy: 0.9593 - auc: 0.9912 - val_loss: 0.7813 - val_accuracy: 0.8621 - val_auc: 0.9262
(83918, 40) (83918,)
(20980, 40) (20980,)


In [11]:
with open("Model_pickle","wb") as f:
  pickle.dump(model,f)

INFO:tensorflow:Assets written to: ram://42a5fa0e-f2c6-4a32-9b12-a35391f9a50b/assets


In [12]:
data= '''6-year-old got gun at LA-area school; families kept in dark

RANCHO CUCAMONGA, Calif. -  Authorities learned a 6-year-old brought a gun to an elementary school outside Los Angeles, but parents of other students didn't learn about it for nearly two weeks.

The student's grandmother found the firearm in his backpack earlier in March. The Sun newspaper reported (goo.gl/3spf8z) that the child said he received it from another student in the Cucamonga School District, about 40 miles (64 kilometers) east of Los Angeles.'''
x_test = _preprocess(data)
y_predict = np.squeeze(model.predict(x_test))
classes_x = (y_predict > 0.5).astype("int32")
print(np.count_nonzero(classes_x==1))
print(np.count_nonzero(classes_x==0))

369
149


In [13]:
from statistics import mode
mode=mode(classes_x)
if(mode==1):
  print("Real")
else:
  print("Fake")

Real
